# 08_02. 현재 필터 알고리즘 프로세싱 문서

이 노트북은 `08_group_camera_feature_ui.py`에 들어간 현재 후처리 흐름을 설명하기 위한 문서이다.

목표는 SegFormer가 만든 scratch mask 또는 `mask_raw_path`를 입력으로 받아서 다음을 확인하는 것이다.

1. mask 내부 픽셀을 K-Means로 나눌 수 있는가?
2. 외곽에 가까운 cluster를 배경으로 보고 제거할 수 있는가?
3. Refinement 또는 JBF를 적용했을 때 점 노이즈가 사라지고 의미 있는 덩어리만 남는가?
4. 최종 mask가 남으면 Alive, 모두 사라지면 Dead로 볼 수 있는가?

아래 코드는 UI 전체를 복제한 것이 아니라, 현재 구현된 알고리즘의 핵심을 설명하기 위해 작게 재현한 확인용 코드이다.

In [ ]:
from pathlib import Path
import math
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

try:
    import cv2
except ImportError as exc:
    raise ImportError('이 노트북은 opencv-python 또는 opencv-contrib-python이 필요합니다.') from exc


def configure_korean_font():
    candidates = ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'AppleGothic']
    available = {font.name for font in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams['font.family'] = name
            break
    plt.rcParams['axes.unicode_minus'] = False


configure_korean_font()
np.set_printoptions(precision=2, suppress=True)
print('cv2 version:', cv2.__version__)
print('OpenCV JBF 사용 가능:', hasattr(cv2, 'ximgproc') and hasattr(cv2.ximgproc, 'jointBilateralFilter'))

## 1. 입력 데이터 구조

현재 UI는 CSV 한 행마다 다음 이미지 경로를 사용할 수 있다.

- `image_path`: 원본 Raw 이미지
- `mask_path`: 배경 0, scratch mask 255인 binary mask
- `mask_raw_path`: 원본 이미지에 mask를 씌우고 mask 밖은 0으로 만든 이미지

알고리즘 관점에서 핵심 입력은 `raw`, `valid_mask`, `mask_raw`이다. 아래에서는 640x640 예시를 직접 만들어서 흐름을 설명한다.

In [ ]:
def show_images(items, cols=3, figsize=(15, 5)):
    rows = int(math.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(figsize[0], figsize[1] * rows))
    axes = np.asarray(axes).reshape(-1)
    for ax in axes:
        ax.axis('off')
    for ax, (title, image, cmap) in zip(axes, items):
        if image.ndim == 2:
            ax.imshow(image, cmap=cmap or 'gray', vmin=0, vmax=255)
        else:
            ax.imshow(np.clip(image, 0, 255).astype(np.uint8))
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def make_demo_sample(size=640, seed=7):
    rng = np.random.default_rng(seed)
    h = w = size

    base = np.zeros((h, w, 3), dtype=np.float32)
    base[..., 0] = 120
    base[..., 1] = 126
    base[..., 2] = 133

    low = rng.normal(0, 1, (h, w, 1)).astype(np.float32)
    low = cv2.GaussianBlur(low, (0, 0), 19)
    if low.ndim == 2:
        low = low[..., None]
    fine = rng.normal(0, 5.5, (h, w, 3)).astype(np.float32)
    raw = base + low * 22 + fine

    # 금속 표면 요철처럼 보이는 약한 선형/점형 노이즈를 추가한다.
    for _ in range(120):
        x0 = int(rng.integers(0, w))
        y0 = int(rng.integers(0, h))
        length = int(rng.integers(8, 55))
        angle = float(rng.uniform(-math.pi, math.pi))
        x1 = int(np.clip(x0 + math.cos(angle) * length, 0, w - 1))
        y1 = int(np.clip(y0 + math.sin(angle) * length, 0, h - 1))
        color = tuple(int(v) for v in rng.normal(0, 6, 3))
        cv2.line(raw, (x0, y0), (x1, y1), color, 1, lineType=cv2.LINE_AA)

    mask = np.zeros((h, w), dtype=np.uint8)
    core = np.zeros((h, w), dtype=np.uint8)
    pts = np.array([[82, 430], [220, 372], [395, 286], [558, 194]], dtype=np.int32)
    cv2.polylines(mask, [pts], False, 255, 23, lineType=cv2.LINE_AA)
    cv2.polylines(core, [pts], False, 255, 13, lineType=cv2.LINE_AA)

    alpha = cv2.GaussianBlur(core.astype(np.float32) / 255.0, (17, 17), 0)[..., None]
    scratch_delta = np.zeros_like(raw)
    scratch_delta[..., 0] = 42
    scratch_delta[..., 1] = 31
    scratch_delta[..., 2] = -18
    raw = raw + alpha * (scratch_delta + rng.normal(0, 7, raw.shape).astype(np.float32))

    raw = np.clip(raw, 0, 255).astype(np.uint8)
    valid_mask = mask > 0
    mask_raw = np.where(valid_mask[..., None], raw, 0).astype(np.uint8)
    return raw, mask, valid_mask, mask_raw


raw, mask, valid_mask, mask_raw = make_demo_sample()
show_images([
    ('Raw 640x640', raw, None),
    ('mask_path 예시', mask, 'gray'),
    ('mask_raw_path 예시', mask_raw, None),
])

## 2. 전처리: Contrast와 Gaussian Blur

선택 패치 분석과 Filter Pipeline에서는 원본 또는 `mask_raw_path`에 대해 간단한 전처리를 먼저 적용할 수 있다.

- Contrast: `(pixel - 127.5) * contrast + 127.5`
- Gaussian blur: 홀수 kernel 크기만 허용

이 단계는 K-Means가 픽셀 그룹을 나누기 전에 색상 차이를 조금 더 선명하게 만들거나, 반대로 점 노이즈를 약화시키기 위한 단계이다.

In [ ]:
def normalize_odd_kernel(value, minimum=1, maximum=31):
    value = int(np.clip(int(value), minimum, maximum))
    if value % 2 == 0:
        value += 1
    return int(np.clip(value, minimum, maximum))


def apply_contrast_and_blur(image, contrast=1.0, blur_kernel=0):
    out = image.astype(np.float32)
    out = (out - 127.5) * float(contrast) + 127.5
    out = np.clip(out, 0, 255).astype(np.uint8)
    if int(blur_kernel) > 1:
        k = normalize_odd_kernel(blur_kernel)
        out = cv2.GaussianBlur(out, (k, k), 0)
    return out


adjusted = apply_contrast_and_blur(raw, contrast=1.35, blur_kernel=3)
show_images([
    ('Raw', raw, None),
    ('Contrast 1.35 + Blur 3', adjusted, None),
    ('전처리 후 mask_raw', np.where(valid_mask[..., None], adjusted, 0), None),
])

## 3. K-Means: mask 내부 픽셀을 색상 그룹으로 분리

현재 구현의 K-Means는 공간 좌표가 아니라 **RGB 픽셀값**만 보고 나눈다.

초기 중심 선택은 다음 순서이다.

1. `valid_mask` 내부 픽셀만 모은다.
2. 최대 25,000개를 seed 17로 샘플링한다.
3. 샘플 픽셀의 luma를 계산한다. `0.299R + 0.587G + 0.114B`
4. `K=2`이면 5%, 95% 분위수에 가까운 실제 픽셀을 초기 center로 잡는다.
5. 이후 일반 K-Means처럼 RGB 거리 기준으로 center를 갱신한다.
6. 마지막에는 center 밝기 순서로 정렬한다. 그래서 보통 `C0`가 어두운 그룹, `C1`이 밝은 그룹이다.

따라서 cluster 경계는 이미지 위의 선이 아니라 RGB 공간에서 가장 가까운 center를 고르는 결정 경계이다.

In [ ]:
LUMA_WEIGHT = np.array([0.299, 0.587, 0.114], dtype=np.float32)


def kmeans_luma_quantile_rgb(image, valid_mask, k=2, max_sample=25_000, seed=17, max_iter=14):
    start = time.perf_counter()
    valid_mask = valid_mask.astype(bool)
    pixels = image[valid_mask].astype(np.float32)
    if len(pixels) == 0:
        return np.full(valid_mask.shape, -1, dtype=np.int16), np.empty((0, 3)), np.array([], dtype=np.int64), 0.0

    k = max(1, min(int(k), len(pixels)))
    if k == 1:
        labels_2d = np.full(valid_mask.shape, -1, dtype=np.int16)
        labels_2d[valid_mask] = 0
        centers = pixels.mean(axis=0, keepdims=True)
        return labels_2d, centers, np.array([len(pixels)], dtype=np.int64), (time.perf_counter() - start) * 1000

    rng = np.random.default_rng(seed)
    sample_size = min(int(max_sample), len(pixels))
    if sample_size == len(pixels):
        sample = pixels
    else:
        sample = pixels[rng.choice(len(pixels), size=sample_size, replace=False)]

    luma = sample @ LUMA_WEIGHT
    centers = []
    for q in np.linspace(0.05, 0.95, k):
        target = np.quantile(luma, q)
        centers.append(sample[int(np.argmin(np.abs(luma - target)))])
    centers = np.asarray(centers, dtype=np.float32)

    for _ in range(max_iter):
        distances = ((sample[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        sample_labels = np.argmin(distances, axis=1)
        new_centers = centers.copy()
        for cluster_id in range(k):
            cluster_pixels = sample[sample_labels == cluster_id]
            if len(cluster_pixels) > 0:
                new_centers[cluster_id] = cluster_pixels.mean(axis=0)
            else:
                new_centers[cluster_id] = sample[rng.integers(0, len(sample))]
        if np.allclose(new_centers, centers, atol=0.5):
            centers = new_centers
            break
        centers = new_centers

    labels = np.empty(len(pixels), dtype=np.int16)
    chunk = 100_000
    for start_idx in range(0, len(pixels), chunk):
        stop_idx = min(start_idx + chunk, len(pixels))
        distances = ((pixels[start_idx:stop_idx, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels[start_idx:stop_idx] = np.argmin(distances, axis=1)

    order = np.argsort(centers @ LUMA_WEIGHT)
    inverse_order = np.empty_like(order)
    inverse_order[order] = np.arange(len(order))
    ordered_labels = inverse_order[labels]
    ordered_centers = centers[order]

    labels_2d = np.full(valid_mask.shape, -1, dtype=np.int16)
    labels_2d[valid_mask] = ordered_labels
    counts = np.bincount(ordered_labels, minlength=k)
    elapsed_ms = (time.perf_counter() - start) * 1000
    return labels_2d, ordered_centers, counts, elapsed_ms


PALETTE = np.array([
    [38, 70, 170],
    [245, 186, 46],
    [220, 70, 90],
    [70, 180, 120],
    [170, 90, 210],
], dtype=np.uint8)


def colorize_labels(labels_2d):
    out = np.zeros((*labels_2d.shape, 3), dtype=np.uint8)
    for cluster_id in range(max(int(labels_2d.max()) + 1, 0)):
        out[labels_2d == cluster_id] = PALETTE[cluster_id % len(PALETTE)]
    return out


labels_2d, centers, counts, kmeans_ms = kmeans_luma_quantile_rgb(adjusted, valid_mask, k=2)
print(f'K-Means time: {kmeans_ms:.1f} ms')
print('centers RGB:', centers)
print('counts:', counts, 'ratio:', counts / counts.sum())

show_images([
    ('mask_raw', mask_raw, None),
    ('K-Means K=2 결과', colorize_labels(labels_2d), None),
    ('K-Means 결과 + Raw overlap', (raw * 0.45 + colorize_labels(labels_2d) * 0.55).astype(np.uint8), None),
])

## 4. 외곽 배경 제거

K-Means를 2개 그룹으로 나누면 어떤 그룹이 scratch이고 어떤 그룹이 배경인지 자동으로 판단해야 한다.

현재 도입한 기준은 다음 가정에 기반한다.

- mask의 바깥쪽 경계에 가까운 픽셀은 실제 scratch 중심보다 배경일 가능성이 높다.
- 따라서 mask 외곽 band에서 많이 등장하는 cluster가 배경일 가능성이 높다.
- 단, 외곽 빈도만 보면 편향될 수 있으므로, cluster center가 외곽 평균 색상과 가까운지도 같이 본다.

점수식은 설명용으로 다음과 같다.

`background_score = 0.75 * border_occupancy + 0.25 * color_similarity`

가장 높은 점수의 cluster를 배경으로 제거하고 나머지를 scratch 후보로 본다.

In [ ]:
def border_band_mask(valid_mask, band_width=3):
    valid = valid_mask.astype(bool)
    k = normalize_odd_kernel(band_width, minimum=1, maximum=31)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    eroded = cv2.erode(valid.astype(np.uint8), kernel, iterations=1).astype(bool)
    band = valid & ~eroded
    return band if band.sum() >= 10 else valid


def infer_border_background_cluster(labels_2d, image, valid_mask, centers):
    band = border_band_mask(valid_mask, band_width=3)
    border_labels = labels_2d[band]
    border_labels = border_labels[border_labels >= 0]
    k = len(centers)
    border_counts = np.bincount(border_labels, minlength=k).astype(np.float32)
    border_occupancy = border_counts / max(float(border_counts.sum()), 1.0)

    border_mean = image[band].astype(np.float32).mean(axis=0)
    center_dist = np.linalg.norm(centers.astype(np.float32) - border_mean[None, :], axis=1)
    max_dist = max(float(center_dist.max()), 1e-6)
    color_similarity = 1.0 - center_dist / max_dist

    score = 0.75 * border_occupancy + 0.25 * color_similarity
    bg_cluster = int(np.argmax(score))
    info = {
        'bg_cluster': bg_cluster,
        'border_occupancy': border_occupancy,
        'color_similarity': color_similarity,
        'score': score,
    }
    return bg_cluster, info


def select_non_background_candidate(labels_2d, image, valid_mask, centers):
    bg_cluster, info = infer_border_background_cluster(labels_2d, image, valid_mask, centers)
    candidate = (labels_2d >= 0) & (labels_2d != bg_cluster) & valid_mask
    return candidate, info


candidate_mask, bg_info = select_non_background_candidate(labels_2d, adjusted, valid_mask, centers)
print('background cluster:', bg_info['bg_cluster'])
print('border occupancy:', bg_info['border_occupancy'])
print('color similarity:', bg_info['color_similarity'])
print('background score:', bg_info['score'])

candidate_rgb = np.zeros_like(raw)
candidate_rgb[candidate_mask] = [255, 60, 60]
show_images([
    ('K-Means 전체', colorize_labels(labels_2d), None),
    ('외곽 배경 제거 후 후보', candidate_mask.astype(np.uint8) * 255, 'gray'),
    ('후보 + Raw overlap', (raw * 0.65 + candidate_rgb * 0.35).astype(np.uint8), None),
])

## 5. Refinement: 형태 기반 정리

Refinement는 K-Means 결과에 남은 점 노이즈를 줄이고 덩어리성을 확인하기 위한 형태학적 후처리이다.

기본 구성은 다음과 같다.

- Morph Open: 작은 점 노이즈 제거
- Morph Close: 끊긴 scratch 후보 연결
- 작은 connected component 제거
- Angle 옵션: component의 bbox 방향을 추정한 뒤, W는 scratch 방향과 평행, H는 수직 방향으로 작동하도록 회전 좌표계에서 morphology를 수행

실제 UI 구현은 component별로 angle-aligned refinement를 수행한다. 아래 코드는 설명을 위한 단일 예시 형태이다.

In [ ]:
def remove_small_components(mask_bool, min_area=12):
    mask_u8 = mask_bool.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    out = np.zeros_like(mask_bool, dtype=bool)
    for label_id in range(1, num_labels):
        if stats[label_id, cv2.CC_STAT_AREA] >= int(min_area):
            out[labels == label_id] = True
    return out


def normalize_axis_angle_deg(angle):
    angle = float(angle)
    while angle <= -90.0:
        angle += 180.0
    while angle > 90.0:
        angle -= 180.0
    return angle


def estimate_component_angle_deg(mask_bool):
    coords_yx = np.column_stack(np.where(mask_bool.astype(bool)))
    if len(coords_yx) < 5:
        return 0.0
    coords_xy = coords_yx[:, ::-1].astype(np.float32)
    rect = cv2.minAreaRect(coords_xy)
    width, height = rect[1]
    angle = float(rect[2])
    if width < height:
        angle += 90.0
    return normalize_axis_angle_deg(angle)


def morphology_clean(mask_bool, kernel_w=3, kernel_h=3, open_iter=1, close_iter=1):
    kw = normalize_odd_kernel(kernel_w)
    kh = normalize_odd_kernel(kernel_h)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, kh))
    work = mask_bool.astype(np.uint8) * 255
    if open_iter:
        work = cv2.morphologyEx(work, cv2.MORPH_OPEN, kernel, iterations=int(open_iter))
    if close_iter:
        work = cv2.morphologyEx(work, cv2.MORPH_CLOSE, kernel, iterations=int(close_iter))
    return work > 0


def rotate_mask(mask_bool, angle_deg):
    h, w = mask_bool.shape
    matrix = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), float(angle_deg), 1.0)
    rotated = cv2.warpAffine(
        mask_bool.astype(np.uint8) * 255,
        matrix,
        (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    return rotated > 0


def morphology_clean_angle_aligned(mask_bool, angle_ref_mask, kernel_w=13, kernel_h=3, open_iter=1, close_iter=1, min_area=12):
    angle_mask = angle_ref_mask.astype(bool) if angle_ref_mask.any() else mask_bool.astype(bool)
    angle = estimate_component_angle_deg(angle_mask)
    rotated = rotate_mask(mask_bool, angle)
    cleaned = morphology_clean(rotated, kernel_w=kernel_w, kernel_h=kernel_h, open_iter=open_iter, close_iter=close_iter)
    restored = rotate_mask(cleaned, -angle)
    restored = remove_small_components(restored & angle_ref_mask.astype(bool), min_area=min_area)
    return restored, angle


refined_plain = remove_small_components(morphology_clean(candidate_mask, kernel_w=5, kernel_h=5), min_area=12)
refined_angle, angle_deg = morphology_clean_angle_aligned(
    candidate_mask,
    valid_mask,
    kernel_w=13,
    kernel_h=3,
    open_iter=1,
    close_iter=1,
    min_area=12,
)
print(f'추정 angle: {angle_deg:.1f} deg')
show_images([
    ('K-Means 후보', candidate_mask.astype(np.uint8) * 255, 'gray'),
    ('일반 Refinement', refined_plain.astype(np.uint8) * 255, 'gray'),
    ('Angle Refinement', refined_angle.astype(np.uint8) * 255, 'gray'),
])

## 6. JBF: Joint Bilateral Filter 기반 mask refinement

JBF는 binary mask 자체만 보는 것이 아니라, Raw 이미지를 guide로 함께 사용한다.

현재 기본 파라미터는 다음과 같다.

- `Diameter = 15`
- `SigmaColor = 30`
- `SigmaSpace = 15`
- `MorphOpen = 3`
- `MorphClose = 5`
- `BlurKernel = 3`
- `Threshold = 230.0 / 255`

처리 순서는 다음이다.

1. cluster mask와 valid mask 주변 ROI만 자른다.
2. cluster mask를 0/255 source로 만든다.
3. Raw 또는 mask_raw를 guide로 하여 `cv2.ximgproc.jointBilateralFilter`를 적용한다.
4. 필요하면 Gaussian blur를 한 번 더 적용한다.
5. threshold 이상인 픽셀만 남긴다.
6. morphology open/close로 정리한다.

즉, JBF는 색상/밝기 경계가 있는 위치는 보존하면서 고립된 mask 점을 약화시키는 목적이다.

In [ ]:
def jbf_refine_mask(
    cluster_mask,
    guide_array,
    valid_mask,
    diameter=15,
    sigma_color=30.0,
    sigma_space=15.0,
    morph_open=3,
    morph_close=5,
    blur_kernel=3,
    threshold=230.0,
):
    start = time.perf_counter()
    cluster_bool = cluster_mask.astype(bool)
    valid_bool = valid_mask.astype(bool)
    roi_mask = valid_bool | cluster_bool
    yy, xx = np.where(roi_mask)
    if len(yy) == 0:
        return cluster_bool, 0.0, 'empty'

    diameter = normalize_odd_kernel(diameter, minimum=1, maximum=31)
    blur_kernel = normalize_odd_kernel(blur_kernel, minimum=1, maximum=31) if int(blur_kernel) > 0 else 0
    threshold_unit = float(np.clip(threshold, 0.0, 255.0)) / 255.0

    h, w = cluster_bool.shape
    pad = max(diameter // 2, blur_kernel // 2, 2)
    y0 = max(int(yy.min()) - pad, 0)
    y1 = min(int(yy.max()) + pad + 1, h)
    x0 = max(int(xx.min()) - pad, 0)
    x1 = min(int(xx.max()) + pad + 1, w)

    source_u8 = (cluster_bool[y0:y1, x0:x1].astype(np.uint8) * 255)
    valid_roi = valid_bool[y0:y1, x0:x1]
    guide_roi = guide_array[y0:y1, x0:x1].astype(np.uint8)

    if hasattr(cv2, 'ximgproc') and hasattr(cv2.ximgproc, 'jointBilateralFilter'):
        filtered_u8 = cv2.ximgproc.jointBilateralFilter(
            guide_roi,
            source_u8,
            diameter,
            float(sigma_color),
            float(sigma_space),
        )
        mode = 'opencv-jbf'
    else:
        # 설명 노트북 실행용 fallback이다. UI의 메인 필터는 OpenCV contrib JBF 사용을 전제로 한다.
        filtered_u8 = cv2.bilateralFilter(source_u8, diameter, max(float(sigma_color), 1e-3), float(sigma_space))
        mode = 'fallback-mask-bilateral'

    filtered = filtered_u8.astype(np.float32) / 255.0
    if blur_kernel > 1:
        filtered = cv2.GaussianBlur(filtered, (blur_kernel, blur_kernel), 0)

    clean_roi = (filtered >= threshold_unit) & valid_roi
    if int(morph_open) or int(morph_close):
        clean_roi = morphology_clean(clean_roi, kernel_w=3, kernel_h=3, open_iter=int(morph_open), close_iter=int(morph_close))

    out = np.zeros_like(cluster_bool, dtype=bool)
    out[y0:y1, x0:x1] = clean_roi & valid_roi
    elapsed_ms = (time.perf_counter() - start) * 1000
    return out, elapsed_ms, mode


jbf_mask, jbf_ms, jbf_mode = jbf_refine_mask(candidate_mask, guide_array=adjusted, valid_mask=valid_mask)
print(f'JBF mode={jbf_mode}, time={jbf_ms:.1f} ms, alive_area={int(jbf_mask.sum())}')
show_images([
    ('K-Means 후보', candidate_mask.astype(np.uint8) * 255, 'gray'),
    ('JBF 결과', jbf_mask.astype(np.uint8) * 255, 'gray'),
    ('JBF 결과 + Raw overlap', (raw * 0.65 + np.dstack([jbf_mask * 255, np.zeros_like(jbf_mask), np.zeros_like(jbf_mask)]).astype(np.uint8) * 0.35).astype(np.uint8), None),
])

## 7. Filter Pipeline과 Alive/Dead 판정

메인 화면의 배치 평가는 하나의 pipeline을 CSV 행마다 적용한다.

현재 pipeline의 큰 구조는 다음과 같다.

```text
source 선택(image_path 또는 mask_raw_path)
  -> contrast / gaussian blur
  -> K-Means on/off
  -> cluster 선택(all, darkest, brightest, largest, smallest, border_background)
  -> JBF와 Refinement 적용 순서 선택
  -> 최종 mask area 계산
  -> area > 0 이면 Alive, area == 0 이면 Dead
```

즉 Alive/Dead는 불량 최종 판정이 아니라, **선택한 후처리 필터를 통과한 mask가 남았는지**를 보는 상태값이다. 이 값이 group별로 어떤 비율을 갖는지 확인해서, 미세스크래치만 효과적으로 제거되는지 검증한다.

In [ ]:
def connected_component_summary(mask_bool):
    clean = mask_bool.astype(bool)
    area = int(clean.sum())
    if area == 0:
        return {
            'area': 0,
            'component_count': 0,
            'largest_component_area': 0,
            'largest_component_ratio': 0.0,
            'largest_bbox_fill_ratio': 0.0,
        }
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(clean.astype(np.uint8), connectivity=8)
    if num_labels <= 1:
        return {'area': area, 'component_count': 0, 'largest_component_area': 0, 'largest_component_ratio': 0.0, 'largest_bbox_fill_ratio': 0.0}
    component_stats = stats[1:]
    largest_idx = int(np.argmax(component_stats[:, cv2.CC_STAT_AREA]))
    largest_area = int(component_stats[largest_idx, cv2.CC_STAT_AREA])
    bbox_area = int(component_stats[largest_idx, cv2.CC_STAT_WIDTH] * component_stats[largest_idx, cv2.CC_STAT_HEIGHT])
    return {
        'area': area,
        'component_count': int(num_labels - 1),
        'largest_component_area': largest_area,
        'largest_component_ratio': largest_area / max(area, 1),
        'largest_bbox_fill_ratio': largest_area / max(bbox_area, 1),
    }


def demo_pipeline(raw, valid_mask, k=2):
    timings = {}
    t0 = time.perf_counter()
    source = apply_contrast_and_blur(raw, contrast=1.35, blur_kernel=3)
    timings['preprocess_ms'] = (time.perf_counter() - t0) * 1000

    labels, centers, counts, kmeans_ms = kmeans_luma_quantile_rgb(source, valid_mask, k=k)
    timings['kmeans_ms'] = kmeans_ms

    candidate, bg_info = select_non_background_candidate(labels, source, valid_mask, centers)

    t1 = time.perf_counter()
    refined, angle = morphology_clean_angle_aligned(candidate, valid_mask, kernel_w=13, kernel_h=3, min_area=12)
    timings['refinement_ms'] = (time.perf_counter() - t1) * 1000

    jbf, jbf_ms, jbf_mode = jbf_refine_mask(refined, guide_array=source, valid_mask=valid_mask)
    timings['jbf_ms'] = jbf_ms

    cc = connected_component_summary(jbf)
    status = 'Alive' if cc['area'] > 0 else 'Dead'
    return {
        'source': source,
        'labels': labels,
        'candidate': candidate,
        'refined': refined,
        'jbf': jbf,
        'status': status,
        'summary': cc,
        'centers': centers,
        'counts': counts,
        'bg_info': bg_info,
        'angle': angle,
        'jbf_mode': jbf_mode,
        'timings': timings,
    }


result = demo_pipeline(raw, valid_mask, k=2)
print('status:', result['status'])
print('angle:', f"{result['angle']:.1f} deg")
print('jbf_mode:', result['jbf_mode'])
print('summary:', result['summary'])
print('timings:', {k: round(v, 1) for k, v in result['timings'].items()})

show_images([
    ('1. 전처리 source', result['source'], None),
    ('2. K-Means 그룹', colorize_labels(result['labels']), None),
    ('3. 외곽 배경 제거 후보', result['candidate'].astype(np.uint8) * 255, 'gray'),
    ('4. Refinement 결과', result['refined'].astype(np.uint8) * 255, 'gray'),
    ('5. JBF 결과', result['jbf'].astype(np.uint8) * 255, 'gray'),
    ('6. 최종 overlay', (raw * 0.65 + np.dstack([result['jbf'] * 255, np.zeros_like(result['jbf']), np.zeros_like(result['jbf'])]).astype(np.uint8) * 0.35).astype(np.uint8), None),
], cols=3, figsize=(15, 4.5))

## 8. 해석 포인트

- K-Means는 위치가 아니라 RGB 색상 기준이다. 그래서 점 노이즈가 많으면 공간적으로 흩어진 결과가 나올 수 있다.
- 외곽 배경 제거는 `mask 내부에서 바깥쪽은 배경일 가능성이 높다`는 가정을 사용한다. mask가 scratch를 너무 타이트하게 잡으면 이 가정이 약해질 수 있다.
- Refinement는 공간적인 덩어리성을 확인하는 단계이다. 얇고 긴 scratch 제거에는 `W/H`와 angle 옵션의 영향이 크다.
- JBF는 Raw의 색상 경계를 guide로 쓰기 때문에 단순 blur보다 mask 경계를 보존한다. 하지만 threshold가 높으면 약한 후보는 쉽게 Dead가 된다.
- 메인 배치의 Alive/Dead는 최종 mask가 남았는지 여부이다. 따라서 group별 Alive 비율을 보면 특정 group이 필터로 얼마나 제거되는지 볼 수 있다.

실제 데이터에서는 이 노트북의 synthetic 예시보다 표면 노이즈와 금속 요철이 강할 수 있으므로, pipeline별로 group/camera 기준 Alive 비율을 비교하는 방식으로 검증해야 한다.